# 00 — Data Pipeline
## BTC/USDT M1 Historical Data — Download, Clean & Resample

---

### Purpose
This notebook is the entry point of the project. It downloads raw M1 OHLCV data
directly from the Bybit public API, validates data quality, resamples to M5 for
the HTF Hull filter, and saves clean CSVs ready for the backtester.

### Why this matters
Most trading strategy projects start from a pre-downloaded CSV. This notebook
shows the full data engineering pipeline — from raw API response to analysis-ready
dataset — which is what production data science actually looks like.

### Bybit API
- Endpoint: `GET /v5/market/kline` (public — no API key required)
- Symbol: `BTCUSDT`
- Interval: `1` (1 minute bars)
- Max per request: 1,000 bars
- Pagination: walk backwards using `end` parameter

### Output files
```
data/raw/       BTCUSDT_M1_raw.csv          raw API response, unmodified
data/processed/ BTCUSDT_M1_clean.csv        validated, UTC-indexed
                BTCUSDT_M5_resampled.csv    M1 resampled to M5 (HTF Hull)
```


---
## Section 1 — Imports & Configuration

### Purpose
Load all libraries and define the download parameters in one place.
All tunable values are marked `← TUNE THIS`.


In [1]:
# ── Standard library ─────────────────────────────────────────────────────────
import time
import os
import json
from datetime import datetime, timezone, timedelta
from pathlib import Path

# ── Data ─────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import requests
from tqdm.notebook import tqdm

# ── Visualisation ─────────────────────────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports OK')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   requests {requests.__version__}')


✅ Imports OK
   pandas  2.3.3
   numpy   2.2.6
   requests 2.34.2


In [2]:
# ── Download configuration ────────────────────────────────────────────────────

# Bybit API
BYBIT_BASE_URL = "https://api.bybit.com"
BYBIT_KLINE_EP = "/v5/market/kline"

# Symbol & timeframe
SYMBOL   = "BTCUSDT"          # ← TUNE THIS: any Bybit spot/perp symbol
INTERVAL = "1"                 # ← TUNE THIS: "1"=M1, "5"=M5, "15"=M15, "60"=H1
CATEGORY = "linear"            # linear = USDT perpetual | spot = spot market

# Date range — match existing M5/M15 data
START_DATE = "2023-01-02"      # ← TUNE THIS: start date (YYYY-MM-DD)
END_DATE   = "2026-05-08"      # ← TUNE THIS: end date   (YYYY-MM-DD)

# API limits
BARS_PER_REQUEST = 1000        # Bybit max bars per request
RATE_LIMIT_PAUSE = 0.2         # seconds between requests (be polite to API)
MAX_RETRIES      = 3           # retries on failed request

# Output paths
RAW_DIR       = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE       = RAW_DIR       / f"{SYMBOL}_M{INTERVAL}_raw.csv"
CLEAN_FILE     = PROCESSED_DIR / f"{SYMBOL}_M{INTERVAL}_clean.csv"
RESAMPLED_FILE = PROCESSED_DIR / f"{SYMBOL}_M5_resampled.csv"

# Convert dates to UTC millisecond timestamps (Bybit uses ms)
start_ts = int(datetime.strptime(START_DATE, "%Y-%m-%d")
               .replace(tzinfo=timezone.utc).timestamp() * 1000)
end_ts   = int(datetime.strptime(END_DATE,   "%Y-%m-%d")
               .replace(tzinfo=timezone.utc).timestamp() * 1000)

# Estimate total bars for progress bar
interval_ms     = int(INTERVAL) * 60 * 1000
total_bars_est  = (end_ts - start_ts) // interval_ms
total_requests  = (total_bars_est // BARS_PER_REQUEST) + 1

print(f'✅ Configuration set')
print(f'   Symbol         : {SYMBOL}')
print(f'   Interval       : M{INTERVAL}')
print(f'   Date range     : {START_DATE} → {END_DATE}')
print(f'   Est. bars      : {total_bars_est:,}')
print(f'   Est. requests  : {total_requests:,}')
print(f'   Raw output     : {RAW_FILE}')
print(f'   Clean output   : {CLEAN_FILE}')
print(f'   Resampled (M5) : {RESAMPLED_FILE}')


✅ Configuration set
   Symbol         : BTCUSDT
   Interval       : M1
   Date range     : 2023-01-02 → 2026-05-08
   Est. bars      : 1,759,680
   Est. requests  : 1,760
   Raw output     : data\raw\BTCUSDT_M1_raw.csv
   Clean output   : data\processed\BTCUSDT_M1_clean.csv
   Resampled (M5) : data\processed\BTCUSDT_M5_resampled.csv


---
## Section 2 — Download Raw M1 Data

### Purpose
Paginate the Bybit `/v5/market/kline` endpoint from END → START (reverse
chronological — Bybit returns most recent bars first within each request).

### Pagination strategy
```
Request 1: end=END_DATE          → bars [END_DATE - 1000min, END_DATE]
Request 2: end=last_bar_start-1  → bars [END_DATE - 2000min, END_DATE - 1000min]
...repeat until start_ts reached
```

### Rate limiting
Bybit allows ~10 requests/second on the public endpoint. We pause
`RATE_LIMIT_PAUSE` seconds between requests to stay well within limits.
Retry logic handles transient failures.


In [3]:
# ── Bybit kline fetcher ───────────────────────────────────────────────────────
def fetch_klines(symbol, interval, category, start_ms, end_ms, limit=1000):
    """
    Fetch one page of klines from Bybit v5 API.

    Parameters
    ----------
    symbol   : str  — e.g. 'BTCUSDT'
    interval : str  — e.g. '1' for M1
    category : str  — 'linear' or 'spot'
    start_ms : int  — start time in milliseconds UTC
    end_ms   : int  — end time in milliseconds UTC
    limit    : int  — bars per request (max 1000)

    Returns
    -------
    list of lists: [[timestamp, open, high, low, close, volume, turnover], ...]
    Most recent bar first.
    """
    params = {
        "category": category,
        "symbol":   symbol,
        "interval": interval,
        "start":    start_ms,
        "end":      end_ms,
        "limit":    limit,
    }
    url = BYBIT_BASE_URL + BYBIT_KLINE_EP

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, params=params, timeout=10)
            resp.raise_for_status()
            data = resp.json()

            if data.get("retCode") != 0:
                raise ValueError(f"Bybit API error: {data.get('retMsg')}")

            return data["result"]["list"]   # list of bars, newest first

        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            print(f"  Retry {attempt}/{MAX_RETRIES}: {e}")
            time.sleep(1.0)

    return []


# ── Main download loop ────────────────────────────────────────────────────────
print(f'Downloading {SYMBOL} M{INTERVAL} data...')
print(f'Range: {START_DATE} → {END_DATE}')
print(f'Estimated {total_requests:,} requests\n')

all_bars  = []
current_end = end_ts
interval_ms = int(INTERVAL) * 60 * 1000

with tqdm(total=total_bars_est, desc="Bars downloaded", unit="bars") as pbar:
    while current_end > start_ts:
        # Calculate window start for this request
        window_start = max(start_ts, current_end - BARS_PER_REQUEST * interval_ms)

        bars = fetch_klines(
            symbol   = SYMBOL,
            interval = INTERVAL,
            category = CATEGORY,
            start_ms = window_start,
            end_ms   = current_end,
            limit    = BARS_PER_REQUEST,
        )

        if not bars:
            break

        all_bars.extend(bars)
        pbar.update(len(bars))

        # Move window back: oldest bar in this batch - 1ms
        oldest_ts = int(bars[-1][0])
        current_end = oldest_ts - 1

        # Stop if we've reached or passed start
        if oldest_ts <= start_ts:
            break

        time.sleep(RATE_LIMIT_PAUSE)

print(f'\n✅ Download complete')
print(f'   Raw bars fetched: {len(all_bars):,}')


Range: 2023-01-02 → 2026-05-08
Estimated 1,760 requests



Bars downloaded:   0%|          | 0/1759680 [00:00<?, ?bars/s]

  Retry 1/3: Bybit API error: svc error: Get kline failed
  Retry 1/3: Bybit API error: svc error: Get kline failed
  Retry 1/3: Bybit API error: svc error: Get kline failed
  Retry 1/3: Bybit API error: Too many visits. Exceeded the API Rate Limit.
  Retry 1/3: Bybit API error: svc error: Get kline failed
  Retry 1/3: Bybit API error: svc error: Get kline failed
  Retry 1/3: Bybit API error: svc error: Get kline failed
  Retry 1/3: Bybit API error: svc error: Get kline failed
  Retry 1/3: Bybit API error: svc error: Get kline failed

✅ Download complete
   Raw bars fetched: 1,759,681


In [4]:
# ── Save raw data ─────────────────────────────────────────────────────────────
# Save the raw API response as-is before any processing
# This is good practice — raw data is immutable, processing is reproducible

raw_df = pd.DataFrame(all_bars, columns=[
    'timestamp_ms', 'open', 'high', 'low', 'close', 'volume', 'turnover'
])

# Bybit returns newest first — sort ascending
raw_df['timestamp_ms'] = raw_df['timestamp_ms'].astype(int)
raw_df = raw_df.sort_values('timestamp_ms').drop_duplicates('timestamp_ms')
raw_df.to_csv(RAW_FILE, index=False)

print(f'✅ Raw data saved → {RAW_FILE}')
print(f'   Rows      : {len(raw_df):,}')
print(f'   Date range: {pd.to_datetime(raw_df["timestamp_ms"].iloc[0], unit="ms", utc=True).date()} '
      f'→ {pd.to_datetime(raw_df["timestamp_ms"].iloc[-1], unit="ms", utc=True).date()}')
print(f'   File size : {RAW_FILE.stat().st_size / 1024 / 1024:.1f} MB')
print()
print('Sample (first 3 rows):')
print(raw_df.head(3).to_string())


✅ Raw data saved → data\raw\BTCUSDT_M1_raw.csv
   Rows      : 1,759,681
   Date range: 2023-01-02 → 2026-05-08
   File size : 110.2 MB

Sample (first 3 rows):
          timestamp_ms     open     high    low  close  volume     turnover
1759680  1672617600000  16622.5  16622.5  16622  16622   38.53   640447.591
1759679  1672617660000    16622  16623.5  16622  16623  16.802   279290.771
1759678  1672617720000    16623    16624  16623  16624  12.271  203990.1315


---
## Section 3 — Clean & Validate

### Purpose
Parse the raw data into a clean, analysis-ready DataFrame and run a suite of
data quality checks. Bad data going into the backtester produces unreliable
results — this section catches issues before they cause silent bugs.

### Checks performed
1. **Timestamp parsing** — convert ms → UTC datetime index
2. **Duplicate bars** — same timestamp appearing twice
3. **Missing bars** — gaps in the M1 sequence (expected around weekends/maintenance)
4. **OHLCV sanity** — high >= low, close/open within high-low range
5. **Zero volume bars** — bars with no trading activity
6. **Gap analysis** — visualise where gaps occur in the timeline


In [5]:
# ── Load raw and parse ────────────────────────────────────────────────────────
raw_df = pd.read_csv(RAW_FILE)
raw_df['timestamp_ms'] = raw_df['timestamp_ms'].astype(int)

# Convert to UTC datetime index
df = raw_df.copy()
df['timestamp'] = pd.to_datetime(df['timestamp_ms'], unit='ms', utc=True)
df = df.set_index('timestamp').sort_index()
df = df[['open','high','low','close','volume']].astype(float)

print(f'✅ Parsed {len(df):,} bars')
print(f'   Start : {df.index[0]}')
print(f'   End   : {df.index[-1]}')
print()
print('Sample:')
print(df.head(3).to_string())


✅ Parsed 1,759,681 bars
   Start : 2023-01-02 00:00:00+00:00
   End   : 2026-05-08 00:00:00+00:00

Sample:
                              open     high      low    close  volume
timestamp                                                            
2023-01-02 00:00:00+00:00  16622.5  16622.5  16622.0  16622.0  38.530
2023-01-02 00:01:00+00:00  16622.0  16623.5  16622.0  16623.0  16.802
2023-01-02 00:02:00+00:00  16623.0  16624.0  16623.0  16624.0  12.271


In [6]:
# ── Data quality checks ───────────────────────────────────────────────────────
print('── Data Quality Report ─────────────────────────────────────────────────')
issues = []

# 1. Duplicates
dupes = df.index.duplicated().sum()
print(f'  Duplicate timestamps  : {dupes:,}')
if dupes > 0:
    df = df[~df.index.duplicated(keep='first')]
    issues.append(f'Removed {dupes} duplicate bars')

# 2. OHLCV sanity checks
bad_hl   = (df['high'] < df['low']).sum()
bad_ch   = (df['close'] > df['high']).sum()
bad_cl   = (df['close'] < df['low']).sum()
bad_oh   = (df['open']  > df['high']).sum()
bad_ol   = (df['open']  < df['low']).sum()
zero_vol = (df['volume'] == 0).sum()

print(f'  high < low            : {bad_hl:,}')
print(f'  close > high          : {bad_ch:,}')
print(f'  close < low           : {bad_cl:,}')
print(f'  open  > high          : {bad_oh:,}')
print(f'  open  < low           : {bad_ol:,}')
print(f'  zero volume bars      : {zero_vol:,}')

# Drop clearly invalid bars
bad_mask = ((df['high'] < df['low']) |
            (df['close'] > df['high']) |
            (df['close'] < df['low']))
if bad_mask.sum() > 0:
    df = df[~bad_mask]
    issues.append(f'Removed {bad_mask.sum()} OHLCV-invalid bars')

# 3. Missing bars — build expected M1 index and compare
expected_idx = pd.date_range(
    start=df.index[0].floor('min'),
    end=df.index[-1].floor('min'),
    freq='1min',
    tz='UTC'
)
missing = expected_idx.difference(df.index)
missing_pct = len(missing) / len(expected_idx) * 100

print(f'\n  Expected M1 bars      : {len(expected_idx):,}')
print(f'  Actual bars           : {len(df):,}')
print(f'  Missing bars          : {len(missing):,}  ({missing_pct:.2f}%)')

# 4. Gap analysis — find consecutive missing bar runs
if len(missing) > 0:
    gap_series = pd.Series(1, index=missing)
    gap_groups = (gap_series.index.to_series().diff() > pd.Timedelta('1min')).cumsum()
    gaps = gap_series.groupby(gap_groups).agg(['count', lambda x: x.index[0], lambda x: x.index[-1]])
    gaps.columns = ['bars_missing', 'gap_start', 'gap_end']
    gaps = gaps.sort_values('bars_missing', ascending=False)
    print(f'\n  Largest gaps:')
    print(gaps.head(10).to_string())

print(f'\n  Issues fixed: {issues if issues else "none"}')
print(f'\n✅ Clean dataset: {len(df):,} bars')

# Bars per day sanity check
days = (df.index[-1] - df.index[0]).days
print(f'   Avg bars/day : {len(df)/days:.0f}  (expect ~1380 on M1 for 23hr crypto market)')


── Data Quality Report ─────────────────────────────────────────────────
  Duplicate timestamps  : 0
  high < low            : 0
  close > high          : 0
  close < low           : 0
  open  > high          : 0
  open  < low           : 0
  zero volume bars      : 1

  Expected M1 bars      : 1,759,681
  Actual bars           : 1,759,681
  Missing bars          : 0  (0.00%)

  Issues fixed: none

✅ Clean dataset: 1,759,681 bars
   Avg bars/day : 1440  (expect ~1380 on M1 for 23hr crypto market)


In [7]:
# ── Gap visualisation ─────────────────────────────────────────────────────────
# Plot monthly bar count to show where data is sparse or missing

monthly_counts = df.resample('ME').size()
expected_per_month = 1380 * 30  # ~1380 bars/day × 30 days

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly_counts.index,
    y=monthly_counts.values,
    name='Actual bars',
    marker_color=['red' if v < expected_per_month * 0.8 else 'steelblue'
                  for v in monthly_counts.values]
))
fig.add_hline(y=expected_per_month * 0.8,
              line_dash='dash', line_color='red',
              annotation_text='80% threshold')

fig.update_layout(
    title=f'{SYMBOL} M1 — Monthly Bar Count (red = >20% data missing)',
    xaxis_title='Month',
    yaxis_title='Bars',
    height=350,
    template='plotly_white',
)
fig.show()


In [8]:
# ── Save clean data ───────────────────────────────────────────────────────────
df.to_csv(CLEAN_FILE)
print(f'✅ Clean data saved → {CLEAN_FILE}')
print(f'   Rows     : {len(df):,}')
print(f'   Size     : {CLEAN_FILE.stat().st_size / 1024 / 1024:.1f} MB')


✅ Clean data saved → data\processed\BTCUSDT_M1_clean.csv
   Rows     : 1,759,681
   Size     : 111.8 MB


---
## Section 4 — Resample M1 → M5

### Purpose
Create the M5 dataset used as the HTF Hull filter in The Trident strategy.

### Resampling rules (standard OHLCV)
```
open   = first bar of the 5-minute window
high   = max of all bars in the window
low    = min of all bars in the window
close  = last bar of the window
volume = sum of all bars in the window
```

### Validation
After resampling we verify:
- Bar count is approximately 1/5 of M1 bars (expect small difference due to gaps)
- No new gaps introduced by the resampling
- OHLCV integrity maintained


In [9]:
# ── Resample M1 → M5 ─────────────────────────────────────────────────────────
df_m5 = df[['open','high','low','close','volume']].resample('5min').agg({
    'open'  : 'first',
    'high'  : 'max',
    'low'   : 'min',
    'close' : 'last',
    'volume': 'sum',
}).dropna()   # drop windows with no bars (gaps)

# ── Validate resampled data ───────────────────────────────────────────────────
print('── M5 Resampled Data Validation ────────────────────────────────────────')
print(f'  M1 bars          : {len(df):,}')
print(f'  M5 bars          : {len(df_m5):,}')
print(f'  Ratio            : {len(df)/len(df_m5):.2f}  (expect ~5.0)')
print()
print(f'  M5 Start         : {df_m5.index[0]}')
print(f'  M5 End           : {df_m5.index[-1]}')
print(f'  M5 Avg bars/day  : {len(df_m5)/days:.0f}  (expect ~276 = 1380/5)')

# OHLCV sanity on resampled
bad_m5 = ((df_m5['high'] < df_m5['low']) |
          (df_m5['close'] > df_m5['high']) |
          (df_m5['close'] < df_m5['low'])).sum()
print(f'  OHLCV violations : {bad_m5}  (expect 0)')

print()
print('Sample (first 3 M5 bars):')
print(df_m5.head(3).to_string())


── M5 Resampled Data Validation ────────────────────────────────────────
  M1 bars          : 1,759,681
  M5 bars          : 351,937
  Ratio            : 5.00  (expect ~5.0)

  M5 Start         : 2023-01-02 00:00:00+00:00
  M5 End           : 2026-05-08 00:00:00+00:00
  M5 Avg bars/day  : 288  (expect ~276 = 1380/5)
  OHLCV violations : 0  (expect 0)

Sample (first 3 M5 bars):
                              open     high      low    close   volume
timestamp                                                             
2023-01-02 00:00:00+00:00  16622.5  16625.0  16622.0  16625.0   94.033
2023-01-02 00:05:00+00:00  16625.0  16627.0  16619.5  16620.0  182.771
2023-01-02 00:10:00+00:00  16620.0  16620.0  16613.5  16614.0  121.639


In [10]:
# ── Save resampled M5 data ────────────────────────────────────────────────────
df_m5.to_csv(RESAMPLED_FILE)
print(f'✅ M5 resampled data saved → {RESAMPLED_FILE}')
print(f'   Rows : {len(df_m5):,}')
print(f'   Size : {RESAMPLED_FILE.stat().st_size / 1024 / 1024:.1f} MB')


✅ M5 resampled data saved → data\processed\BTCUSDT_M5_resampled.csv
   Rows : 351,937
   Size : 23.3 MB


---
## Section 5 — Summary & Next Steps

### Purpose
Final sanity check on all outputs and a quick visual of the price series
to confirm the data looks correct before passing to the backtester.


In [11]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('── Pipeline Summary ────────────────────────────────────────────────────')
print()

files = {
    'Raw M1'        : RAW_FILE,
    'Clean M1'      : CLEAN_FILE,
    'Resampled M5'  : RESAMPLED_FILE,
}
for name, path in files.items():
    if path.exists():
        size = path.stat().st_size / 1024 / 1024
        rows = sum(1 for _ in open(path)) - 1  # subtract header
        print(f'  {name:<18}: {rows:>8,} rows  |  {size:.1f} MB  →  {path}')
    else:
        print(f'  {name:<18}: ❌ NOT FOUND')

print()
print('── Ready for backtester ────────────────────────────────────────────────')
print(f'  Load M1  : pd.read_csv("{CLEAN_FILE}", index_col=0, parse_dates=True)')
print(f'  Load M5  : pd.read_csv("{RESAMPLED_FILE}", index_col=0, parse_dates=True)')


── Pipeline Summary ────────────────────────────────────────────────────

  Raw M1            : 1,759,681 rows  |  110.2 MB  →  data\raw\BTCUSDT_M1_raw.csv
  Clean M1          : 1,759,681 rows  |  111.8 MB  →  data\processed\BTCUSDT_M1_clean.csv
  Resampled M5      :  351,937 rows  |  23.3 MB  →  data\processed\BTCUSDT_M5_resampled.csv

── Ready for backtester ────────────────────────────────────────────────
  Load M1  : pd.read_csv("data\processed\BTCUSDT_M1_clean.csv", index_col=0, parse_dates=True)
  Load M5  : pd.read_csv("data\processed\BTCUSDT_M5_resampled.csv", index_col=0, parse_dates=True)


In [12]:
# ── Price chart — visual sanity check ─────────────────────────────────────────
# Sample 5000 M1 bars for a readable chart (full dataset is too dense)
sample = df.iloc[::max(1, len(df)//5000)]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.75, 0.25],
                    subplot_titles=[f'{SYMBOL} M1 Price', 'Volume'])

fig.add_trace(go.Candlestick(
    x=sample.index,
    open=sample['open'], high=sample['high'],
    low=sample['low'],   close=sample['close'],
    name='Price',
    increasing_line_color='#1D9E75',
    decreasing_line_color='#E24B4A',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=sample.index,
    y=sample['volume'],
    name='Volume',
    marker_color='rgba(100,100,200,0.5)',
), row=2, col=1)

fig.update_layout(
    title=f'{SYMBOL} M1 — {START_DATE} → {END_DATE} ({len(df):,} bars)',
    height=500,
    template='plotly_white',
    xaxis_rangeslider_visible=False,
    showlegend=False,
)
fig.show()
print('✅ Pipeline complete — ready to build 01_trident_backtest.ipynb')


✅ Pipeline complete — ready to build 01_trident_backtest.ipynb
